# 🤖 Interactive Chat & Code Playground with Mirendil-152M Champion ($M_{156}$)

Welcome! In this notebook, you can interact directly with our **152M-parameter Recursive Self-Improvement (RSI) Champion Model** ($M_{156}$), which achieved **58.2% Pass@1** and **61.8% Pass@5** across 156 autonomous generations on our DGX Blackwell GB10 GPU.

---

## 🎯 Model Capabilities: What to Expect

| Capability Category | Status | Notes |
| :--- | :---: | :--- |
| **Core Logic & Conditionals** | ⭐⭐⭐⭐⭐ **100%** | Deterministic on `is_even`, `is_positive`, `negate_boolean`. |
| **Dictionaries & Hash Maps** | ⭐⭐⭐⭐☆ **80%** | Word counts, frequency maps, key lookups. |
| **Strings & Text Processing** | ⭐⭐⭐⭐☆ **70%** | Palindromes, reversals (`s[::-1]`), vowel counting, capitalization. |
| **Lists & Arrays** | ⭐⭐⭐⭐☆ **70%** | Extraction, filtering, finding min/max, slicing. |
| **Mathematics** | ⭐⭐⭐☆☆ **60%** | Factorials, power, absolute values, sums. |
| **Standard Algorithms** | ⭐⭐☆☆☆ **40%** | Bubble sort, binary search, linear search. |
| **Conversational Chit-Chat** | ⚠️ *Base Model* | This is a **code foundation model**, not an RLHF chatbot like ChatGPT. When prompted with conversational instructions, it uses a few-shot code template to return working Python code! |

---

## Step 1: Load the Champion Model ($M_{156}$) & Tokenizer

Run this cell to load the 152M model into memory (runs on GPU if available, or CPU in ~500 MB RAM).

In [1]:
import sys
from pathlib import Path

# Set repository root
root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
import torch.nn.functional as F
from src.model.transformer import TransformerDecoder, ModelConfig
from src.tokenizer.bpe import CodeTokenizer

# Auto-detect device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: [{device}]")

# Locate champion checkpoint
ckpt_candidates = [
    root_dir / "checkpoints" / "152m_champion_m156.pt",
    root_dir / "checkpoints" / "152m_wsd_10m_baseline" / "best_model.pt",
    Path("checkpoints/152m_champion_m156.pt"),
]

ckpt_path = None
for c in ckpt_candidates:
    if c.exists():
        ckpt_path = c
        break

if not ckpt_path:
    raise FileNotFoundError(
        "Champion checkpoint not found! Please verify checkpoints/152m_champion_m156.pt exists."
    )

print(f"Loading checkpoint: {ckpt_path}...")
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
config = ckpt.get("config") or ModelConfig()

model = TransformerDecoder(config)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval()

tokenizer = CodeTokenizer()
print(f"\n Model loaded successfully!")
print(f"  - Parameters: 152 Million (Tied Embeddings)")
print(f"  - Context Window: {config.max_seq_len} tokens")
print(f"  - Architecture: Pre-RMSNorm + RoPE + SwiGLU + GQA (12 Q : 4 KV)")

Using compute device: [cpu]
Loading checkpoint: m:\Rama\Study\llm\LLM\RSI\checkpoints\152m_champion_m156.pt...

 Model loaded successfully!
  - Parameters: 152 Million (Tied Embeddings)
  - Context Window: 2048 tokens
  - Architecture: Pre-RMSNorm + RoPE + SwiGLU + GQA (12 Q : 4 KV)


---
## Step 2: Define Autoregressive Sampling Engine

This cell sets up temperature and nucleus (top-p) sampling with stop-word handling.

In [2]:
def top_p_sampling(
    logits: torch.Tensor, temperature: float = 0.6, top_p: float = 0.95
) -> torch.Tensor:
    if temperature <= 0.01:
        return torch.argmax(logits, dim=-1, keepdim=True)
    logits = logits / temperature
    probs = F.softmax(logits, dim=-1)
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    sorted_indices_to_remove = cumulative_probs > top_p
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = False
    indices_to_remove = sorted_indices_to_remove.scatter(
        1, sorted_indices, sorted_indices_to_remove
    )
    probs = probs.masked_fill(indices_to_remove, 0.0)
    probs = probs / probs.sum(dim=-1, keepdim=True)
    return torch.multinomial(probs, num_samples=1)


@torch.no_grad()
def complete_code(
    prompt: str,
    max_new_tokens: int = 96,
    temperature: float = 0.2,
    top_p: float = 0.95,
    stop_tokens: list = None,
) -> str:
    token_ids = tokenizer.encode(prompt)
    x = torch.tensor(token_ids, dtype=torch.long, device=device).unsqueeze(0)
    generated_ids = []

    for _ in range(max_new_tokens):
        idx_cond = (
            x
            if x.size(1) <= model.config.max_seq_len
            else x[:, -model.config.max_seq_len :]
        )
        logits, _ = model(idx_cond)
        next_token_logits = logits[:, -1, :]
        next_token = top_p_sampling(
            next_token_logits, temperature=temperature, top_p=top_p
        )
        token_val = next_token.item()

        if token_val == tokenizer.eos_id:
            break
        generated_ids.append(token_val)
        x = torch.cat((x, next_token), dim=1)

        current_text = tokenizer.decode(generated_ids)
        if stop_tokens:
            for stop_word in stop_tokens:
                if stop_word in current_text:
                    return current_text.split(stop_word)[0]

    return tokenizer.decode(generated_ids)


print("Generation engine ready!")

Generation engine ready!


---
## Mode 1: Raw Python Code Completion Playground

Type any Python function signature or docstring below and watch the model complete the implementation!

In [ ]:
# Example 1: String processing
prompt = "def is_palindrome(s: str) -> bool:"
print(f"Prompt:\n{prompt}")
print("-" * 40)
completion = complete_code(prompt, max_new_tokens=64, temperature=0.2)
print(f"Model Output:\n{prompt}{completion}")

In [ ]:
# Example 2: Mathematics
prompt = "def factorial(n: int) -> int:"
print(f"Prompt:\n{prompt}")
print("-" * 40)
completion = complete_code(prompt, max_new_tokens=64, temperature=0.2)
print(f"Model Output:\n{prompt}{completion}")

In [ ]:
# Example 3: Dictionary frequency mapping
prompt = "def count_char_frequencies(text: str) -> dict:"
print(f"Prompt:\n{prompt}")
print("-" * 40)
completion = complete_code(prompt, max_new_tokens=80, temperature=0.2)
print(f"Model Output:\n{prompt}{completion}")

---
## Mode 2: Conversational Coding Chat (Instruction-Wrapped)

Ask the model natural language programming questions using our few-shot assistant wrapper:

In [ ]:
CHAT_TEMPLATE = '''"""
Python Programming Assistant.
Answers coding queries with clean, idiomatic Python code.
"""

# User: Write a function to check if a number is positive.
# Assistant:
def is_positive(n):
    return n > 0

# User: Write a function to multiply two numbers.
# Assistant:
def multiply_two_numbers(a, b):
    return a * b

# User: {query}
# Assistant:
'''

def ask_assistant(user_query: str, temperature: float = 0.3) -> str:
    full_prompt = CHAT_TEMPLATE.format(query=user_query)
    raw_ans = complete_code(
        full_prompt,
        max_new_tokens=100,
        temperature=temperature,
        stop_tokens=["# User:", "<|endoftext|>"]
    )
    return raw_ans.strip()

# Try a conversational query:
query = "Write a Python function to reverse a list in-place without using list.reverse()"
print(f"User: {query}\n")
ans = ask_assistant(query)
print(f"Assistant:\n{ans}")

---
## Mode 3: Live Interactive Chat Loop

Run the cell below to start an interactive terminal chat session directly in the notebook output! Type your questions or coding tasks, or type `'quit'` to exit.

In [ ]:
print("🤖 Mirendil-152M Chat Assistant Live! (Type 'quit' or 'exit' to stop)")
print("=" * 65)

while True:
    try:
        user_q = input("You > ").strip()
        if not user_q or user_q.lower() in ('quit', 'exit', 'q'):
            print("\nSession ended. Thanks for chatting!")
            break
            
        # Check if user typed raw code or conversational query
        if user_q.startswith("def ") or user_q.startswith("class "):
            resp = complete_code(user_q, max_new_tokens=96, temperature=0.2)
            print(f"\nModel (Code Completion) >\n{user_q}{resp}\n")
        else:
            resp = ask_assistant(user_q, temperature=0.3)
            print(f"\nModel (Assistant) >\n{resp}\n")
    except KeyboardInterrupt:
        print("\nChat stopped.")
        break